In [1]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import pandas as pd
from collections import defaultdict
import os

In [2]:
def import_dataset():
    file_root = Path('../../data/processed/combined/')
    data = []

    for split_path in file_root.glob("*.txt"):
        with open(split_path, "r") as file:
            for line in file:
                img_path, seg_map_path = line.strip().split()
                data.append({
                    "image_path": 'C:/Users/varno/Documents/Masters/THESIS/offroad_terrain_segmentation/data/processed/combined/' + str(img_path),
                    "annotation_path": 'C:/Users/varno/Documents/Masters/THESIS/offroad_terrain_segmentation/data/processed/combined/' + str(seg_map_path)
                })

    df = pd.DataFrame(data)

    return df

In [59]:
def class_distribution(df):
    class_dist_count = defaultdict(int)

    id_to_name = {
        0: 'sky',
        1: 'Traversable',
        2: 'Non-Traversable',
        3: 'Object'
    }

    for path in df["annotation_path"]:
        with Image.open(path) as img:
            mask_array = np.array(img)

        ids, counts = np.unique(mask_array, return_counts=True)

        for i in range(len(ids)):
            id = ids[i]
            class_dist_count[id] += counts[i]
        
    print(f"Number of unique colors: {len(class_dist_count)}")
    print("\n")

    total = sum(class_dist_count.values())

    print("\n=== CLASS DISTRIBUTION (% & counts) ===")

    found_ids = []
    found_counts = []
    found_percentages= []

    for id, count in sorted(class_dist_count.items(), key=lambda x: x[1], reverse=True):
        pct = (count / total) * 100
        print(f"{id_to_name[id]}: {pct:.2f}% ({count})")
        found_ids.append(id_to_name[id])
        found_counts.append(count)
        found_percentages.append(round(pct, 2))

    return found_ids, found_counts, found_percentages

In [60]:
df = import_dataset()
print(len(df))
found_ids, found_counts, found_percentages = class_distribution(df)

13393
Number of unique colors: 4



=== CLASS DISTRIBUTION (% & counts) ===
Traversable: 39.86% (6591957234)
sky: 25.47% (4211871661)
Object: 19.75% (3267147974)
Non-Traversable: 14.92% (2467733531)


In [69]:
import statistics
total_pixels = 0

for count in found_counts:
    total_pixels += count

median = round(statistics.median(sorted(found_percentages)), 2)

weights = defaultdict(int)
for i, percent in enumerate(found_percentages):
    class_name = found_ids[i]
    weight = round(median / percent, 2)
    weights[class_name] = weight

for name, distribution in sorted(weights.items()):
        print(f"{name}: {distribution}")

Non-Traversable: 1.52
Object: 1.14
Traversable: 0.57
sky: 0.89
